<a href="https://colab.research.google.com/github/davidrpugh/data-analytics-with-polars/blob/main/ch09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 9: Combining Expressions

This notebook accompanies Chapter 9 of *Python Polars: The Definitive Guide* by Jeroen Janssens and Thijs Nieuwdorp. It is adapted for **Polars 2.1** and **Google Colab**.

In Chapter 7 you learned to *begin* expressions and in Chapter 8 to *continue* them. This chapter is about **combining** them. You need to combine expressions whenever the column you want depends on **more than one** value or column: the ratio of two columns, a filter with several conditions, a full name built from first and last names.

By the end of this notebook you will be able to:

- combine expressions with **inline operators** (`+`, `>`, `&`, ...) and with the equivalent **methods**, and know when each is clearer
- write **arithmetic**, **comparison**, **Boolean**, and **bitwise** combinations, and avoid their classic pitfalls (operator precedence, chained comparisons, `and`/`or`)
- understand how combinations treat **missing values** (`null`)
- use functions that combine **many** columns at once: `concat_str`, `all_horizontal`, `sum_horizontal`, `coalesce`, and more
- write conditional logic with `pl.when(...).then(...).otherwise(...)`

**How to use this notebook:** run the cells from top to bottom. Several cells *intentionally* trigger errors to teach a point.

## 0. Setup

We pin Polars to the **2.1 series** so everyone sees the same behavior, and use `%pip` so the package is installed into the running notebook kernel.

In [ ]:
%pip install -q "polars==2.1.*"

### Imports and helpers

- `download` fetches a data file from the book's repository **only if it isn't already there**. It writes to a temporary name and renames when complete, and raises on HTTP errors instead of saving an error page as if it were data.

In [ ]:
import math
from pathlib import Path
import urllib.request

from IPython.display import display
import polars as pl
import polars.selectors as cs
from polars.testing import assert_frame_equal


print(f"Polars {pl.__version__}")


BASE_URL = "https://github.com/davidrpugh/data-analytics-with-polars/raw/refs/heads/main/data"
DATA_DIR = Path("data")


def download(relative_path: str) -> Path:
    """Download BASE_URL/relative_path into DATA_DIR (unless it exists) and return the local path."""
    path = DATA_DIR / relative_path
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_name(path.name + ".tmp")
        urllib.request.urlretrieve(f"{BASE_URL}/{relative_path}", temporary)
        temporary.rename(path)
    return path


### The data

We start with the ten-fruit table from Chapter 7.

In [ ]:
FRUIT_CSV = download("fruit.csv")

fruit = pl.read_csv(FRUIT_CSV)
fruit

Here is an example you already know: keep the fruit that is **round** *and* **heavier than 1,000 grams**. It combines three things: an existing column, a column compared with a value, and a Boolean combination of the two.

1. `pl.col("weight") > 1000` combines a column and a value. It produces a Boolean Series (`True` where the weight is over 1,000).
2. `&` combines that Series with the Boolean column `is_round`. The result is `True` only where *both* are `True`.
3. `filter` keeps the rows where the final Series is `True`.

In [ ]:
fruit.filter(pl.col("is_round") & (pl.col("weight") > 1000))

### Operator precedence: why those parentheses matter

Python decides in which order to apply operators using fixed **precedence** rules. The key fact for this chapter: the Boolean operators `&`, `|`, and `^` bind **more tightly** than comparisons like `>` and `==`. So without the parentheses around `pl.col("weight") > 1000`, Python groups the operators in a way you didn't intend:

In [ ]:
fruit.filter(pl.col("is_round") & pl.col("weight") > 1000)

Python read that as `(pl.col("is_round") & pl.col("weight")) > 1000`, which tries to combine a Boolean column with a number. **Rule: wrap every comparison in parentheses before combining it with `&`, `|`, or `^`.**

## Inline Operators Versus Methods

There are two ways to combine expressions:

- an **inline operator**: `pl.col("i") * pl.col("j")`
- a **method**: `pl.col("i").mul(pl.col("j"))`

They produce exactly the same result. We recommend the **inline operator** when one exists, because the spaces around it make it obvious that you're combining two expressions. The trade-off is that to keep chaining methods (such as `alias`) after an inline combination, you must wrap it in **parentheses**. Methods chain naturally, but read less like math.

In [ ]:
combine_df = pl.DataFrame(
    {
        "i": [6.0, 0, 2, 2.5],
        "j": [7.0, 1, 2, 3]
    }
).with_columns(
    (pl.col("i") * pl.col("j")).alias("*"),
    pl.col("i").mul(pl.col("j")).alias("Expr.mul()"),
)

In [ ]:
combine_df

Every inline operator has a method; the reverse isn't true (for instance there is no operator for the dot product). Here is the full mapping for this chapter:

| Kind | Operator | Method |
|---|---|---|
| Arithmetic | `+` `-` `*` `/` | `add` `sub` `mul` `truediv` |
| Arithmetic | `//` `**` `%` | `floordiv` `pow` `mod` |
| Arithmetic | *(none)* | `dot` |
| Comparison | `<` `<=` `==` | `lt` `le` `eq` |
| Comparison | `>=` `>` `!=` | `ge` `gt` `ne` |
| Boolean | `&` `\|` `~` `^` | `and_` `or_` `not_` `xor` |

The Boolean methods end in an underscore because `and`, `or`, and `not` are reserved words in Python.


## Arithmetic Operations

Arithmetic works between two expressions, or between an expression and a plain Python value. Here we convert the weight from grams to kilograms. We give the result an honest name with a keyword, since a column called `weight` that holds kilograms would be misleading:

In [ ]:
fruit.select("name", weight_kg=pl.col("weight") / 1000)

All the arithmetic operators in one table. Polars creates a float column automatically when needed (for instance for `/`).

To keep the table narrow we set two display options **temporarily**, with a context manager. Inside the `with` block the settings apply; afterwards, Polars goes back to its defaults, so the options can't leak into later cells. We use `display` *inside* the block, because a DataFrame that is shown at the end of a cell is rendered *after* the block has already closed.

In [ ]:
arithmetic_df = pl.DataFrame(
    {
        "i": [0.0, 2, 2, -2, -2],
        "j": [1, 2, 3, 4, -5]
    }
).with_columns(
    (pl.col("i") + pl.col("j")).alias("i + j"),
    (pl.col("i") - pl.col("j")).alias("i - j"),
    (pl.col("i") * pl.col("j")).alias("i * j"),
    (pl.col("i") / pl.col("j")).alias("i / j"),
    (pl.col("i") // pl.col("j")).alias("i // j"),
    (pl.col("i") ** pl.col("j")).alias("i ** j"),
    (pl.col("j") % 2).alias("j % 2"),
)

with pl.Config(float_precision=2, tbl_cell_numeric_alignment="RIGHT", tbl_cols=-1):
    display(arithmetic_df)

**Floor division and modulo with negative numbers.** `//` rounds *down* (toward negative infinity), not toward zero, and `%` is defined to go with it. Python does the same. Compare Polars with Python on a few cases rather than guess:

In [ ]:
pairs = [(7, 2), (-7, 2), (7, -2), (-7, -2)]

pl.DataFrame(
    {
        "i": [i for i, _ in pairs],
        "j": [j for _, j in pairs],
        "python_floordiv": [i // j for i, j in pairs],
        "python_mod": [i % j for i, j in pairs],
    }
).with_columns(
    polars_floordiv=pl.col("i") // pl.col("j"),
    polars_mod=pl.col("i") % pl.col("j"),
)

**Dividing by zero** doesn't crash a query; the result for that row is special (infinity or `NaN` for floats). For *integer* floor division, dividing by zero returns `null`.

In [ ]:
zero_df = pl.DataFrame({"i": [1, 0, -1], "j": [0, 0, 0]})

zero_df.select(division=pl.col("i") / pl.col("j"))

In [ ]:
zero_df.select(floor_division=pl.col("i") // pl.col("j"))

**The dot product** has no inline operator. It multiplies two columns element by element and **sums** the results into a *single* number. Because the answer is one value, it is repeated on every row if you add it next to the original columns.

In [ ]:
dot_df = arithmetic_df.select("i", "j", dot=pl.col("i").dot(pl.col("j")))


In [ ]:
dot_df

## Comparison Operations

Comparisons answer yes/no questions: *Which experiments were significant? Which movies scored above 8.7? Are these voltages in range?* A comparison always produces a **Boolean Series**, which you can add as a column, use to `filter`, or use in `pl.when` (at the end of this notebook).

You can compare numbers, strings, and temporal types. Strings are compared **alphabetically** (lexicographically), which has a famous pitfall: the text `"10"` sorts *before* `"9"`, because `"1"` comes before `"9"`.

In [ ]:
pl.select(
    a_before_b=pl.lit("a") < pl.lit("b"),
    ten_before_nine_as_text=pl.lit("10") < pl.lit("9"),
    ten_before_nine_as_numbers=pl.lit(10) < pl.lit(9),
)

In [ ]:
fruit.select("name", "weight").filter(pl.col("weight") >= 1000)

### Chaining comparisons doesn't work, and what to do instead

In plain Python you can write a **chained comparison**:

In [ ]:
x = 4
3 < x < 5

Python expands that into `(3 < x) and (x < 5)`. The `and` needs to turn each side into a single `True` or `False`, but a Polars expression is a *description of a whole column*, so it has no single truth value. Let's see what Polars says:

In [ ]:
pl.select(pl.lit(3) < pl.lit(x) < pl.lit(5))

The same problem occurs if you use the Python keywords `and` or `or` anywhere you combine expressions. Always use `&` and `|`.

There are two ways to write the range test correctly.

**Option 1:** two comparisons joined with `&`, each in parentheses:

In [ ]:
pl.select((pl.lit(3) < pl.lit(x)) & (pl.lit(x) < pl.lit(5))).item()

**Option 2:** the dedicated method `is_between`. It's shorter, but be careful: **by default it is *inclusive* at both ends** (`closed="both"`), whereas `3 < x < 5` is *exclusive*. They agree for `x = 4` but disagree at the boundary, `x = 3`:

In [ ]:
boundary = pl.DataFrame({"x": [2, 3, 4, 5, 6]}).with_columns(
    strict=(pl.lit(3) < pl.col("x")) & (pl.col("x") < pl.lit(5)),
    is_between_default=pl.col("x").is_between(3, 5),
    is_between_none=pl.col("x").is_between(3, 5, closed="none"),
    is_between_left=pl.col("x").is_between(3, 5, closed="left"),
    is_between_right=pl.col("x").is_between(3, 5, closed="right"),
)

boundary

`closed` takes `"both"` (the default), `"left"`, `"right"`, or `"none"`. Choose it deliberately: boundary mistakes are a classic source of off-by-one bugs.


### Comparing floating-point numbers

Be careful with `==` on floats, because tiny rounding errors make "equal" numbers unequal. Instead, test whether the **difference is small enough** ("close to"). In the table below, compare the exact `b == a` with the tolerance-based test for `b ≈ √2`. This cell also combines arithmetic and comparison in one expression, and applies one comparison to **all** columns with `pl.all()` and `name.suffix`.

In [ ]:
(
    pl.DataFrame(
        {"a": [-273.15, 0, 42, 100], "b": [1.4142, 2.7183, 42, 3.1415]}
    ).with_columns(
        (pl.col("a") == pl.col("b")).alias("a == b"),
        (pl.col("a") <= pl.col("b")).alias("a <= b"),
        (pl.all() > 0).name.suffix(" > 0"),
        (pl.col("b").is_close(pl.lit(2).sqrt(), abs_tol=0, rel_tol=1e-9)).alias("b ≈ √2"),
        ((1 < pl.col("b")) & (pl.col("b") < 3)).alias("1 < b < 3"),
    )
)

### Comparing missing values

A comparison with a `null` doesn't produce `True` or `False`. It produces **`null`** ("unknown"). That matters in `filter`, which keeps only rows where the condition is `True`, so rows with a missing value silently **disappear**:

In [ ]:
with_missing = pl.DataFrame({"x": [1, None, 3]})

with_missing.with_columns(
    equals_none=pl.col("x") == None,       # noqa: E711, deliberately comparing with None
    greater_than_2=pl.col("x") > 2,
    is_null=pl.col("x").is_null(),
)

In [ ]:
with_missing.filter(pl.col("x") != 3)

The row with the missing value is **not** in the result of `x != 3`, even though "missing" is arguably "not 3". If you want `null` to count as a regular value in the comparison, use the null-aware versions `eq_missing` and `ne_missing` (or test explicitly with `is_null`):

In [ ]:
with_missing.with_columns(
    eq_3=pl.col("x").eq_missing(3),
    ne_3=pl.col("x").ne_missing(3),
)

In [ ]:
with_missing.filter(pl.col("x").ne_missing(3))

### Which types can be compared?

Not every pair of types can be compared. Each comparison below is run on its own, and we record either its result or the kind of error. Read the table carefully, because the **result** column holds the *answer* to the comparison (or the error), not whether it was "allowed". (The book labels a similar table "allowed", but the values in it are comparison results.)

The rows marked "(after cast)" and "(after extracting the time)" show the standard fix for an incompatible pair: convert one side first. Note that `"5" < "3"` is **a text comparison** (false) once the number has been cast to a string.

In [ ]:
comparisons = {
    "Boolean > integer": pl.lit(True) > 0,
    "time > time": pl.time(23, 58) > pl.time(0, 0),
    "datetime < date": pl.datetime(1969, 7, 21, 2, 56) < pl.date(1976, 7, 20),
    "string < integer": pl.lit("5") < pl.lit(3),
    "string < integer (after cast)": pl.lit("5") < pl.lit(3).cast(pl.String),
    "datetime > time": pl.datetime(1999, 1, 1) > pl.time(0, 0),
    "datetime != time (after extracting the time)": pl.datetime(1999, 1, 1).dt.time() != pl.time(0, 0),
}

rows = []
for label, expression in comparisons.items():
    try:
        outcome = str(pl.select(expression).item())
    except Exception as error:
        outcome = f"error: {type(error).__name__}"
    rows.append({"comparison": label, "result": outcome})

pl.DataFrame(rows)

## Boolean Algebra Operations

Boolean algebra combines True/False values. Three basic operations are enough to build all the others:

| Operation | Operator | True when... |
|---|---|---|
| **Conjunction** (AND) | `&` | *both* sides are True |
| **Disjunction** (OR) | `\|` | *at least one* side is True |
| **Negation** (NOT) | `~` | the value is False |

Polars adds one derived operation, **exclusive or** (`^`, XOR): True when *exactly one* side is True. Others are easy to build from these: NAND is `~(p & q)` and NOR is `~(p | q)`.

Here's the earlier range test as Boolean algebra. With `x = 7`, `p` is True and `q` is False, so `p & q` is False:

In [ ]:
value = 7
p = pl.lit(3) < pl.lit(value)   # True
q = pl.lit(value) < pl.lit(5)   # False
pl.select(p & q).item()

A **truth table** lists the result of every combination. Compare each column with its definition above.

In [ ]:
truth_table = pl.DataFrame(
    {"p": [True, True, False, False], "q": [True, False, True, False]}
).with_columns(
    (pl.col("p") & pl.col("q")).alias("p & q"),
    (pl.col("p") | pl.col("q")).alias("p | q"),
    (~pl.col("p")).alias("~p"),
    (pl.col("p") ^ pl.col("q")).alias("p ^ q"),
    (~(pl.col("p") & pl.col("q"))).alias("NAND"),
    (~(pl.col("p") | pl.col("q"))).alias("NOR"),
)

truth_table

The method forms (`and_`, `or_`, `not_`, `xor`) give the same answers.

### The Python keywords `and`, `or`, `not` don't work

Python's `and`, `or`, and `not` try to turn an expression into a single True/False, which isn't possible for a whole column (the same reason chained comparisons fail). Use `&`, `|`, and `~`:

In [ ]:
truth_table.select(pl.col("p") and pl.col("q"))

### Boolean logic with missing values

What is "unknown AND False"? It must be **False**, since the result is False no matter what the unknown turns out to be. And "unknown OR True" must be **True**. This is called *three-valued* (Kleene) logic, and Polars follows it. Only when the unknown could change the answer does the result stay `null`. Check the table against that reasoning:

In [ ]:
pl.DataFrame(
    {
        "p": [True, True, True, False, False, False, None, None, None],
        "q": [True, False, None, True, False, None, True, False, None],
    }
).with_columns(
    and_=pl.col("p") & pl.col("q"),
    or_=pl.col("p") | pl.col("q"),
    not_p=~pl.col("p"),
)

## Bitwise Operations

When you apply `&`, `|`, `^`, and `~` to **integers** instead of Booleans, they work on the integers' **bits**, one position at a time. For example, `10 | 34` is `00001010 | 00100010 = 00101010`, which is 42:

In [ ]:
pl.select(pl.lit(10) | pl.lit(34)).item()

We use unsigned 8-bit integers (`UInt8`) so the bit patterns are easy to read. Look at the `~x` column: negation flips **all eight bits**, so `~1` is not `0`, it is `254`, because flipping `00000001` gives `11111110`.

In [ ]:
bits = pl.DataFrame(
    {"x": [1, 1, 0, 0, 7, 10], "y": [1, 0, 1, 0, 2, 34]},
    schema={"x": pl.UInt8, "y": pl.UInt8},
).with_columns(
    (pl.col("x") & pl.col("y")).alias("x & y"),
    (pl.col("x") | pl.col("y")).alias("x | y"),
    (~pl.col("x")).alias("~x"),
    (pl.col("x") ^ pl.col("y")).alias("x ^ y"),
)

bits

To *see* the bits, we convert each number to a binary string with a Python format string. (This uses a Python function on every value, which is slow on large data, see Chapter 17. It's fine for a small teaching table.) We verify the result against plain Python:

In [ ]:
binary = bits.select(pl.all().map_elements("{0:08b}".format, return_dtype=pl.String))

assert binary["x | y"].to_list() == [format(a | b, "08b") for a, b in zip(bits["x"], bits["y"])]
binary

**Booleans or ones and zeros?** The result of `&`, `|`, `^` is the same whether you use Booleans or `1`/`0`, *except for negation*: `~True` is `False`, but `~1` (as an unsigned byte) is `254`, not `0`. Booleans are also more compact (one bit per value, versus eight for a `UInt8`). **Use Booleans whenever a column has only two possible values.**

In [ ]:
flags = pl.DataFrame({"flag": [True, False], "bit": [1, 0]}, schema_overrides={"bit": pl.UInt8})

flags.select(not_flag=~pl.col("flag"), not_bit=~pl.col("bit"))

In [ ]:
many = [True, False] * 500_000

print(f"Boolean: {pl.Series(many).estimated_size('kb'):,.0f} KB")
print(f"UInt8:   {pl.Series(many).cast(pl.UInt8).estimated_size('kb'):,.0f} KB")

## Using Functions

Some combinations involve **many** expressions, or have no operator. For these there are **module-level functions** (called as `pl.something(...)`). Here are the ones you'll use most:

| Function | Combines the inputs into... |
|---|---|
| `pl.concat_str`, `pl.format` | one **String** |
| `pl.concat_list`, `pl.struct` | one **List** / one **Struct** |
| `pl.all_horizontal`, `pl.any_horizontal` | one Boolean (AND / OR across columns) |
| `pl.sum_horizontal`, `pl.min_horizontal`, `pl.max_horizontal` | one number (across columns) |
| `pl.coalesce` | the first value that isn't `null` |
| `pl.corr`, `pl.cov` | a correlation or covariance of two columns |
| `pl.when(...)` | a conditional (see the next section) |

"Horizontal" means *across the columns of each row*, in contrast to the usual "vertical" operations *down* a column. We use a table of famous scientists.

In [ ]:
scientists = pl.DataFrame(
    {
        "first_name": ["George", "Grace", "John", "Kurt", "Ada"],
        "last_name": ["Boole", "Hopper", "Tukey", "Gödel", "Lovelace"],
        "country": ["England", "United States", "United States", "Austria-Hungary", "England"],
    }
)
scientists

### Combining into a List or a Struct

`concat_list` puts the values of several columns into one **List** per row (all of the same type). `pl.struct` puts them into one **Struct** with named fields, so they can have different types.

We select the name columns with a **selector** (`cs.ends_with("_name")`) rather than a regular expression. (The book writes `pl.col("^*_name$")`, a regex that happens to work but has an odd leading `^*`.) A struct can be expanded back into columns with `unnest`.

In [ ]:
scientists.select(
    concat_list=pl.concat_list(cs.ends_with("_name")),
    struct=pl.struct(pl.all()),
)

In [ ]:
scientists.select(pl.struct(pl.all())).unnest("first_name")

### Combining into a String

`concat_str` joins columns with a separator. `format` fills `{}` placeholders in a template, which gives you more control over the layout:

In [ ]:
scientists.select(
    concat_str=pl.concat_str(pl.all(), separator=" "),
    format=pl.format("{}, {} from {}", "last_name", "first_name", "country"),
)

**Missing values in `concat_str`.** By default, if *any* part is `null`, the whole result is `null`. You can skip the missing parts instead with `ignore_nulls=True`:

In [ ]:
people = pl.DataFrame({"first": ["Ada", "Alan"], "middle": ["Augusta", None], "last": ["Lovelace", "Turing"]})

people.select(
    default=pl.concat_str("first", "middle", "last", separator=" "),
    ignore_nulls=pl.concat_str("first", "middle", "last", separator=" ", ignore_nulls=True),
)

### Across the columns of a row: `all_horizontal` and `any_horizontal`

These are like `&` and `|`, but across *as many columns as you like*, without typing them all out. Here we ask: does each person like **everything**, or **at least one thing**?

We select the "likes" columns by name with a selector (`cs.starts_with("likes_")`), so adding another `likes_...` column later requires no changes. We give the results descriptive names (the book uses `all` and `any`, which hide Python's built-in functions of the same names).

In [ ]:
prefs = pl.DataFrame(
    {
        "id": [1, 7, 42, 101, 999],
        "has_pet": [True, False, True, False, True],
        "likes_travel": [False, False, False, False, True],
        "likes_movies": [True, False, True, False, True],
        "likes_books": [False, False, True, True, True],
    }
).with_columns(
    likes_all=pl.all_horizontal(cs.starts_with("likes_")),
    likes_any=pl.any_horizontal(cs.starts_with("likes_")),
)

prefs

### Across the columns of a row: sums, minima, and maxima

`sum_horizontal`, `min_horizontal`, and `max_horizontal` work across columns. They work on Booleans too, where `True` counts as 1, so summing Booleans **counts how many are True**. (The book applies them to *all* columns, including the `id` column, which makes the numbers meaningless. Here we select only the Boolean columns.)

In [ ]:
prefs.select(
    "id",
    n_true=pl.sum_horizontal(cs.boolean()),
    n_likes=pl.sum_horizontal(cs.starts_with("likes_")),
)

For numbers, the same functions give a row-wise total, lowest, and highest. Notice what happens to a missing value (check the documentation for how `ignore_nulls` is set in your version):

In [ ]:
scores = pl.DataFrame({"math": [85, 78, None], "science": [90, 82, 77], "history": [88, 80, 91]})

scores.select(
    total=pl.sum_horizontal(pl.all()),
    lowest=pl.min_horizontal(pl.all()),
    highest=pl.max_horizontal(pl.all()),
)

### The first non-missing value: `coalesce`

`coalesce` scans its arguments left to right and returns the first value that isn't `null`. It's the standard way to say "use column A, and if that is missing, fall back to column B, then to a default".

In [ ]:
contacts = pl.DataFrame(
    {
        "name": ["Ada", "Alan", "Grace"],
        "mobile": ["555-0100", None, None],
        "home": [None, "555-0111", None],
    }
)

contacts.with_columns(phone=pl.coalesce("mobile", "home", pl.lit("no phone")))

### Correlation of two columns

`pl.corr` computes the Pearson correlation of two columns (and `pl.cov` the covariance). A perfect straight line gives exactly 1; a perfect inverse line gives -1.

In [ ]:
pl.DataFrame({"x": [1, 2, 3, 4, 5], "up": [2, 4, 6, 8, 10], "down": [10, 8, 6, 4, 2]}).select(
    corr_up=pl.corr("x", "up"),
    corr_down=pl.corr("x", "down"),
)

### When, Then, Otherwise

The function that deserves special attention is `pl.when`, a **vectorized `if`**: for every row, it checks a condition and picks a value. It works like Python's `if` / `elif` / `else`:

- `pl.when(condition).then(value)` starts the chain
- you can add more `.when(...).then(...)` branches
- `.otherwise(value)` supplies the value for rows where **no condition** was true

**Branches are checked in order, and the first True wins**, so put the most specific condition first. Here we classify each person: do they like everything, something, or nothing?

In [ ]:
prefs.select(
    "id",
    likes_what=pl.when(pl.all_horizontal(cs.starts_with("likes_")))
    .then(pl.lit("Likes everything"))
    .when(pl.any_horizontal(cs.starts_with("likes_")))
    .then(pl.lit("Likes something"))
    .otherwise(pl.lit("Likes nothing")),
)

**Order matters.** If we check "likes something" *first*, everyone who likes everything is also caught by that branch, so the "everything" branch never fires:

In [ ]:
prefs.select(
    "id",
    wrong_order=pl.when(pl.any_horizontal(cs.starts_with("likes_")))
    .then(pl.lit("Likes something"))
    .when(pl.all_horizontal(cs.starts_with("likes_")))
    .then(pl.lit("Likes everything"))
    .otherwise(pl.lit("Likes nothing")),
)

**Literal text needs `pl.lit`.** In `then` and `otherwise`, a plain Python **string is interpreted as a column name**, not as text. So `.then("Flagged")` looks for a column called `Flagged`. Always wrap literal text in `pl.lit(...)`:

In [ ]:
orders = pl.DataFrame(
    {
        "order_amount": [500, 750, 1200, 800, 1100],
        "status": ["Approved", "Processing", "Processing", "Declined", "Processing"],
    }
)

orders.with_columns(flag=pl.when(pl.col("order_amount") > 1000).then("Flagged"))

### What if there is no `otherwise`?

If no condition matches and there is no `otherwise`, the value is **`null`**. A frequent surprise is that this *overwrites* the existing column. Suppose orders over 1,000 need flagging. The version below flags them, but also replaces every other `status` with `null`:

In [ ]:
orders.with_columns(
    status=pl.when(pl.col("order_amount") > 1000).then(pl.lit("Flagged"))
)

In plain Python, an `if` without an `else` leaves things as they are, but here you must say so. Pass the original column to `otherwise`:

In [ ]:
orders.with_columns(
    status=pl.when(pl.col("order_amount") > 1000)
    .then(pl.lit("Flagged"))
    .otherwise(pl.col("status"))
)

The `then` and `otherwise` values can be **any expression**, including calculations. Here a different fee rate applies depending on the amount:

In [ ]:
orders.with_columns(
    fee=pl.when(pl.col("order_amount") > 1000)
    .then(pl.col("order_amount") * 0.02)
    .otherwise(pl.col("order_amount") * 0.01)
)

**Missing conditions.** If the condition itself is `null` (because of missing data), the row doesn't count as "True", so it falls through to the next branch or the `otherwise`. Check:

In [ ]:
pl.DataFrame({"amount": [500, None, 1200]}).with_columns(
    size=pl.when(pl.col("amount") > 1000).then(pl.lit("large")).otherwise(pl.lit("not large"))
)

Notice that the missing amount is labeled "not large". Whether that is the right answer is up to you. If "unknown" should stay unknown, add an explicit branch first, for example `pl.when(pl.col("amount").is_null()).then(pl.lit("unknown"))`. A conditional that quietly files unknowns under "otherwise" is a common source of subtle bugs.

## Takeaways

- Combine expressions with **inline operators** (clearer) or **methods** (chainable). Both give the same result. After an inline operator, use parentheses to keep chaining.
- **Parenthesize every comparison** before combining it with `&`, `|`, or `^`, because those bind more tightly than comparisons.
- Python's **chained comparisons** and the keywords **`and` / `or` / `not`** don't work on expressions. Use two comparisons joined with `&`, or `is_between`, and `&`, `|`, `~`.
- `is_between` is **inclusive** by default. Set `closed` deliberately.
- A comparison with `null` gives `null`, and `filter` drops those rows. Use `eq_missing` / `ne_missing` or `is_null` when `null` should count.
- Boolean logic with `null` follows **three-valued** logic: `null & False` is False, `null | True` is True.
- On integers the same operators work **bit by bit**; use Booleans for two-valued columns.
- **Horizontal functions** (`all_horizontal`, `sum_horizontal`, `concat_str`, `coalesce`, ...) combine many columns per row. Select the columns with **selectors**.
- `pl.when(...).then(...).otherwise(...)` is a vectorized `if`. Wrap literal text in `pl.lit`, order the branches carefully, and remember that a missing `otherwise` means `null`.
- Use `with pl.Config(...)` for temporary display settings so they can't leak into other cells.

### Next up

In Chapter 10 we start Part IV and put expressions to work **transforming** DataFrames: selecting and creating columns, with column selectors.

### Try it yourself


1. Filter `fruit` to fruit that is **orange** *or* **heavier than 2,000 grams**, but **not** round. Write it once with operators and once with the `and_`/`or_`/`not_` methods, and check they agree.


2. Add a column `size` to `fruit` with `pl.when`: `"tiny"` under 10 g, `"small"` under 200 g, `"large"` otherwise. Then swap the order of the first two branches and explain what changes.

3. Create a column `label` that is `name` in capitals, followed by `(origin)` in parentheses, using `pl.format`.

4. Use `pl.coalesce` to fill missing values of one column from another, then check with `null_count()` that none remain.

5. For `x = [1, 2, 3, 4, 5]`, compare `is_between(2, 4)` with `closed="left"`, and list which values differ from the default.